In [1]:
import os
import cv2
import math
import pandas as pd
import mediapipe as mp
from pathlib import Path

PROJECT_ROOT = Path("..")

LANDMARKS_DIR = PROJECT_ROOT / "data" / "alphabet" / "landmarks"

TRAIN_LANDMARKS_CSV = LANDMARKS_DIR / "train_landmarks_normalized.csv"
TEST_LANDMARKS_CSV = LANDMARKS_DIR / "test_landmarks_normalized.csv"

TRAIN_FAILED_CSV = LANDMARKS_DIR / "train_failed.csv"
TEST_FAILED_CSV = LANDMARKS_DIR / "test_failed.csv"

TRAIN_RECOVERED_CSV = LANDMARKS_DIR / "train_recovered.csv"
TEST_RECOVERED_CSV = LANDMARKS_DIR / "test_recovered.csv"

TRAIN_STILL_FAILED_CSV = LANDMARKS_DIR / "train_still_failed.csv"
TEST_STILL_FAILED_CSV = LANDMARKS_DIR / "test_still_failed.csv"

TRAIN_MERGED_CSV = LANDMARKS_DIR / "train_landmarks_merged.csv"
TEST_MERGED_CSV = LANDMARKS_DIR / "test_landmarks_merged.csv"

mp_hands = mp.solutions.hands

print("Train failed exists:", TRAIN_FAILED_CSV.exists())
print("Test failed exists:", TEST_FAILED_CSV.exists())

Train failed exists: True
Test failed exists: True


In [2]:
def normalize_landmarks(landmarks):
    wrist = landmarks[0]

    shifted = []

    for x, y, z in landmarks:
        shifted.append((x - wrist[0], y - wrist[1], z - wrist[2]))

    max_dist = 0.0

    for x, y, z in shifted:
        dist = math.sqrt(x**2 + y**2 + z**2)

        if dist > max_dist:
            max_dist = dist

    if max_dist == 0:
        return None

    normalized = []

    for x, y, z in shifted:
        normalized.extend([
            x / max_dist,
            y / max_dist,
            z / max_dist
        ])

    return normalized

In [3]:
def rotate_image(image, angle, bg_color=(255, 255, 255)):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)

    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

    rotated = cv2.warpAffine(
        image,
        matrix,
        (w, h),
        borderValue=bg_color
    )

    return rotated


def improve_contrast(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l = clahe.apply(l)

    merged = cv2.merge((l, a, b))
    improved = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

    return improved


def build_retry_candidates(image):
    candidates = []

    candidates.append(image)

    # Small rotations help MediaPipe detect hands that are angled slightly differently.
    for angle in [-20, -15, -10, -5, 5, 10, 15, 20]:
        candidates.append(rotate_image(image, angle))

    # Slight padding can help if the hand is close to image borders.
    for pad in [20, 40, 80]:
        padded = cv2.copyMakeBorder(
            image,
            pad,
            pad,
            pad,
            pad,
            borderType=cv2.BORDER_CONSTANT,
            value=(255, 255, 255)
        )

        candidates.append(padded)

    # Contrast version.
    candidates.append(improve_contrast(image))

    # Contrast + rotations.
    contrast = improve_contrast(image)

    for angle in [-15, -10, -5, 5, 10, 15]:
        candidates.append(rotate_image(contrast, angle))

    return candidates

In [4]:
def extract_landmarks_retry(image_path, hands):
    image = cv2.imread(str(image_path))

    if image is None:
        return None

    candidates = build_retry_candidates(image)

    for img in candidates:
        img = cv2.resize(img, (512, 512))
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            coords = [
                (lm.x, lm.y, lm.z)
                for lm in results.multi_hand_landmarks[0].landmark
            ]

            normalized = normalize_landmarks(coords)

            if normalized is not None:
                return normalized

    return None

In [5]:
def retry_failed_files(failed_csv_path, split_name):
    failed_df = pd.read_csv(failed_csv_path)

    recovered_rows = []
    still_failed = []

    columns = []

    for i in range(21):
        columns.extend([f"x{i}", f"y{i}", f"z{i}"])

    columns += ["label", "file_path", "split"]

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.1
    ) as hands:

        for index, row in failed_df.iterrows():
            label = row["label"]
            file_path = row["file_path"]

            print(f"Retrying {split_name}: {label} | {index + 1}/{len(failed_df)}")

            features = extract_landmarks_retry(file_path, hands)

            if features is not None:
                recovered_rows.append(
                    features + [label, file_path, split_name]
                )
            else:
                still_failed.append({
                    "label": label,
                    "file_path": file_path,
                    "split": split_name
                })

    recovered_df = pd.DataFrame(recovered_rows, columns=columns)
    still_failed_df = pd.DataFrame(still_failed)

    return recovered_df, still_failed_df

In [6]:
train_recovered, train_still_failed = retry_failed_files(
    TRAIN_FAILED_CSV,
    "train"
)

print("Recovered train:", len(train_recovered))
print("Still failed train:", len(train_still_failed))

display(train_recovered["label"].value_counts().sort_index())

Retrying train: A | 1/5663
Retrying train: A | 2/5663
Retrying train: A | 3/5663
Retrying train: A | 4/5663
Retrying train: A | 5/5663
Retrying train: A | 6/5663
Retrying train: A | 7/5663
Retrying train: A | 8/5663
Retrying train: A | 9/5663
Retrying train: A | 10/5663
Retrying train: A | 11/5663
Retrying train: A | 12/5663
Retrying train: A | 13/5663
Retrying train: A | 14/5663
Retrying train: A | 15/5663
Retrying train: A | 16/5663
Retrying train: A | 17/5663
Retrying train: A | 18/5663
Retrying train: A | 19/5663
Retrying train: A | 20/5663
Retrying train: A | 21/5663
Retrying train: A | 22/5663
Retrying train: A | 23/5663
Retrying train: A | 24/5663
Retrying train: A | 25/5663
Retrying train: A | 26/5663
Retrying train: A | 27/5663
Retrying train: A | 28/5663
Retrying train: A | 29/5663
Retrying train: A | 30/5663
Retrying train: A | 31/5663
Retrying train: A | 32/5663
Retrying train: A | 33/5663
Retrying train: A | 34/5663
Retrying train: A | 35/5663
Retrying train: A | 36/5663
R

label
A     44
B    152
C     63
D     98
E     93
F     94
G     57
H     71
I     77
K     90
L     76
M    139
N    121
O    161
P    105
Q    108
R    104
S    116
T     85
U     91
V    177
W    141
X    220
Y     74
Name: count, dtype: int64

In [7]:
test_recovered, test_still_failed = retry_failed_files(
    TEST_FAILED_CSV,
    "test"
)

print("Recovered test:", len(test_recovered))
print("Still failed test:", len(test_still_failed))

display(test_recovered["label"].value_counts().sort_index())

Retrying test: A | 1/1105
Retrying test: A | 2/1105
Retrying test: A | 3/1105
Retrying test: A | 4/1105
Retrying test: A | 5/1105
Retrying test: A | 6/1105
Retrying test: A | 7/1105
Retrying test: A | 8/1105
Retrying test: A | 9/1105
Retrying test: A | 10/1105
Retrying test: A | 11/1105
Retrying test: A | 12/1105
Retrying test: A | 13/1105
Retrying test: A | 14/1105
Retrying test: A | 15/1105
Retrying test: A | 16/1105
Retrying test: A | 17/1105
Retrying test: A | 18/1105
Retrying test: A | 19/1105
Retrying test: A | 20/1105
Retrying test: A | 21/1105
Retrying test: A | 22/1105
Retrying test: A | 23/1105
Retrying test: A | 24/1105
Retrying test: A | 25/1105
Retrying test: A | 26/1105
Retrying test: A | 27/1105
Retrying test: A | 28/1105
Retrying test: A | 29/1105
Retrying test: A | 30/1105
Retrying test: A | 31/1105
Retrying test: A | 32/1105
Retrying test: A | 33/1105
Retrying test: A | 34/1105
Retrying test: A | 35/1105
Retrying test: B | 36/1105
Retrying test: B | 37/1105
Retrying t

label
A    35
B    41
C    20
D    33
E    18
F    24
G    11
H    13
I    33
K    26
L    21
M    32
N    15
O    38
P    18
Q    26
R    15
S    32
T    16
U    14
V    29
W    39
X    21
Y    27
Name: count, dtype: int64

In [8]:
train_recovered.to_csv(TRAIN_RECOVERED_CSV, index=False)
test_recovered.to_csv(TEST_RECOVERED_CSV, index=False)

train_still_failed.to_csv(TRAIN_STILL_FAILED_CSV, index=False)
test_still_failed.to_csv(TEST_STILL_FAILED_CSV, index=False)

print("Saved recovered and still failed files.")

Saved recovered and still failed files.


Merge files

In [11]:
import pandas as pd
from pathlib import Path

LANDMARKS_DIR = Path("../data/alphabet/landmarks")

TRAIN_ORIGINAL_CSV = LANDMARKS_DIR / "train_landmarks_normalized.csv"
TEST_ORIGINAL_CSV = LANDMARKS_DIR / "test_landmarks_normalized.csv"

TRAIN_RECOVERED_CSV = LANDMARKS_DIR / "train_recovered.csv"
TEST_RECOVERED_CSV = LANDMARKS_DIR / "test_recovered.csv"

TRAIN_MERGED_CSV = LANDMARKS_DIR / "train_landmarks_merged.csv"
TEST_MERGED_CSV = LANDMARKS_DIR / "test_landmarks_merged.csv"

In [12]:
old_train = pd.read_csv(TRAIN_ORIGINAL_CSV)
old_test = pd.read_csv(TEST_ORIGINAL_CSV)

train_recovered = pd.read_csv(TRAIN_RECOVERED_CSV)
test_recovered = pd.read_csv(TEST_RECOVERED_CSV)

print("Old train:", old_train.shape)
print("Recovered train:", train_recovered.shape)

print("Old test:", old_test.shape)
print("Recovered test:", test_recovered.shape)

display(train_recovered.head())

Old train: (4827, 66)
Recovered train: (2557, 66)
Old test: (695, 66)
Recovered test: (597, 66)


,x0,y0,z0,x1,y1,z1,x2,y2,z2,x3,...,z18,x19,y19,z19,x20,y20,z20,label,file_path,split
0,0.0,0.0,0.0,0.285368,-0.068278,-0.108837,0.479958,-0.358031,-0.098570,0.523657,...,-0.034633,-0.150768,-0.459212,-0.005858,-0.121773,-0.371319,0.051351,A,..\data\alphabet\raw\Train\A\Image_1685009090....,train
1,0.0,0.0,0.0,0.276934,-0.015804,-0.112554,0.563446,-0.177691,-0.152988,0.729408,...,-0.168995,-0.053897,-0.617619,-0.152229,-0.057923,-0.478312,-0.108751,A,..\data\alphabet\raw\Train\A\Image_1685009110....,train
2,0.0,0.0,0.0,0.242485,-0.089151,-0.162419,0.462569,-0.372220,-0.222907,0.529170,...,-0.268437,-0.253627,-0.557318,-0.246682,-0.215522,-0.401327,-0.189140,A,..\data\alphabet\raw\Train\A\Image_1685009112....,train
3,0.0,0.0,0.0,0.248450,-0.047931,-0.157168,0.417369,-0.404674,-0.188744,0.397502,...,-0.195642,-0.282554,-0.380954,-0.156044,-0.245831,-0.318320,-0.099116,A,..\data\alphabet\raw\Train\A\Image_1685009115....,train
4,0.0,0.0,0.0,0.229720,-0.155490,-0.090136,0.392308,-0.471711,-0.096296,0.416128,...,-0.115413,-0.353574,-0.727209,-0.111410,-0.294900,-0.593096,-0.077748,A,..\data\alphabet\raw\Train\A\Image_1685009120....,train


In [13]:
merged_train = pd.concat(
    [old_train, train_recovered],
    ignore_index=True
)

merged_test = pd.concat(
    [old_test, test_recovered],
    ignore_index=True
)

merged_train = merged_train.drop_duplicates(subset=["file_path", "split"])
merged_test = merged_test.drop_duplicates(subset=["file_path", "split"])

merged_train.to_csv(TRAIN_MERGED_CSV, index=False)
merged_test.to_csv(TEST_MERGED_CSV, index=False)

print("Merged train:", merged_train.shape)
print("Merged test:", merged_test.shape)

print("Saved:")
print(TRAIN_MERGED_CSV)
print(TEST_MERGED_CSV)

Merged train: (7384, 66)
Merged test: (1292, 66)
Saved:
..\data\alphabet\landmarks\train_landmarks_merged.csv
..\data\alphabet\landmarks\test_landmarks_merged.csv


In [14]:
print("Merged train distribution:")
display(merged_train["label"].value_counts().sort_index())

print("Merged test distribution:")
display(merged_test["label"].value_counts().sort_index())

Merged train distribution:


label
A    434
B    283
C    203
D    284
E    322
F    279
G    407
H    382
I    275
K    364
L    276
M    251
N    351
O    302
P    372
Q    367
R    183
S    286
T    353
U    230
V    265
W    215
X    347
Y    353
Name: count, dtype: int64

Merged test distribution:


label
A    75
B    61
C    39
D    42
E    62
F    66
G    68
H    68
I    69
K    49
L    69
M    46
N    71
O    51
P    57
Q    68
R    21
S    62
T    72
U    15
V    37
W    40
X    24
Y    60
Name: count, dtype: int64

In [15]:
def oversample_training_data(df):
    max_count = df["label"].value_counts().max()

    balanced_parts = []

    for label, group in df.groupby("label"):
        sampled = group.sample(
            n=max_count,
            replace=True,
            random_state=42
        )

        balanced_parts.append(sampled)

    balanced_df = pd.concat(balanced_parts, ignore_index=True)

    balanced_df = balanced_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    return balanced_df

In [16]:
TRAIN_BALANCED_CSV = LANDMARKS_DIR / "train_landmarks_balanced.csv"

balanced_train = oversample_training_data(merged_train)

balanced_train.to_csv(TRAIN_BALANCED_CSV, index=False)

print("Balanced train:", balanced_train.shape)
print("Saved:", TRAIN_BALANCED_CSV)

display(balanced_train["label"].value_counts().sort_index())

Balanced train: (10416, 66)
Saved: ..\data\alphabet\landmarks\train_landmarks_balanced.csv


label
A    434
B    434
C    434
D    434
E    434
F    434
G    434
H    434
I    434
K    434
L    434
M    434
N    434
O    434
P    434
Q    434
R    434
S    434
T    434
U    434
V    434
W    434
X    434
Y    434
Name: count, dtype: int64